# Reinicio del Data Mart con Python

Este notebook documenta cómo eliminar y reconstruir las tablas del Data Mart
(esquema estrella: 1 tabla de hechos + 6 dimensiones + 1 staging) usando Python.

**Motor:** Microsoft SQL Server / SQL Server LocalDB

**Propósito:** Permitir un "reset" limpio del Data Mart durante el desarrollo
del ETL en SSIS, sin borrar la base de datos completa.

---

## 1. ¿Qué hace este notebook?

Ejecuta el script `src/etl/reset_datamart.py` que:

1. **Conecta** a la base `ECBD_Compradores` usando `pyodbc`.
2. **Elimina** las 8 tablas en orden correcto (respetando claves foráneas):
   - `FactSesion` (tiene 6 FK) → primeras 6 dimensiones → `StgSesiones`.
3. **Recrea** las 8 tablas con exactamente la misma definición que el script SQL
   original `sql/ddl/01_crear_base_y_datamart.sql`.
4. **Valida** que todo se haya creado correctamente (tablas, PK, FK, UNIQUE).
5. Es **idempotente**: se puede ejecutar las veces que se quiera sin errores.

---

## 2. ¿Por qué es necesario?

- Durante el desarrollo del ETL en **SSIS (Visual Studio)** es frecuente
  necesitar vaciar y recrear las tablas para probar cargas limpias.
- Permite restablecer el Data Mart a un estado conocido **sin borrar la base
  de datos completa** (ahorra tiempo y evita problemas de permisos).
- Al estar en **Python**, se puede integrar fácilmente en pipelines de CI/CD,
  llamar desde notebooks o automatizar en scripts de despliegue.
- Sirve como **"reset" antes de cada ejecución de prueba** del paquete SSIS.

---

## 3. Requisitos previos

| Requisito | Comando / Acción |
|-----------|------------------|
| Python 3.8+ | Ya instalado en el entorno del proyecto |
| `pyodbc` | `pip install pyodbc` (ya en requirements.txt) |
| SQL Server LocalDB corriendo | `sqllocaldb start MSSQLLocalDB` |
| Base `ECBD_Compradores` existe | Ejecutar `sql/ddl/01_crear_base_y_datamart.sql` una vez |

---

## 4. Ejecución del script

### Opción A: Desde terminal (recomendado)

```bash
# Desde la raíz del proyecto
python -m src.etl.reset_datamart
```

### Opción B: Desde este notebook

Ejecuta la celda siguiente:

In [1]:
import subprocess
import sys
import os

# Añadir la raíz del proyecto al PYTHONPATH para que encuentre src.etl
raiz_proyecto = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if raiz_proyecto not in sys.path:
    sys.path.insert(0, raiz_proyecto)

# Ejecutar el script de reinicio desde la raíz del proyecto
resultado = subprocess.run(
    [sys.executable, "-m", "src.etl.reset_datamart"],
    capture_output=True,
    text=True,
    cwd=raiz_proyecto
)

print("STDOUT:")
print(resultado.stdout)
print("\nSTDERR:")
print(resultado.stderr)
print(f"\nCódigo de salida: {resultado.returncode}")

STDOUT:
REINICIO DEL DATA MART - ECBD_Compradores

Conectando a: (localdb)\MSSQLLocalDB / ECBD_Compradores

[ERROR] ERROR DE CONEXIÃ“N: ('08001', '[08001] [Microsoft][ODBC Driver 17 for SQL Server]SQL Server Network Interfaces:  (-1983577846) (SQLDriverConnect); [08001] [Microsoft][ODBC Driver 17 for SQL Server]Login timeout expired (0); [08001] [Microsoft][ODBC Driver 17 for SQL Server]A network-related or instance-specific error has occurred while establishing a connection to SQL Server. Server is not found or not accessible. Check if instance name is correct and if SQL Server is configured to allow remote connections. For more information see SQL Server Books Online. (-1983577846)')
  Verifica que:
  - SQL Server LocalDB estÃ© corriendo (sqllocaldb start MSSQLLocalDB)
  - La base ECBD_Compradores exista (ejecuta 01_crear_base_y_datamart.sql primero)
  - pyodbc estÃ© instalado (pip install pyodbc)


STDERR:


Código de salida: 1


---

## 5. Qué esperar en la salida

Si todo va bien, verás algo como:

```
======================================================================
REINICIO DEL DATA MART - ECBD_Compradores
======================================================================

Conectando a: (localdb)\MSSQLLocalDB / ECBD_Compradores
  ✓ Conexión establecida

--- ELIMINANDO TABLAS EXISTENTES ---
  → Eliminando Tabla de hechos (tiene 6 FK hacia dimensiones) (dbo.FactSesion)
    ✓ dbo.FactSesion eliminada
  → Eliminando Dimensión Mes (dbo.DimMes)
    ✓ dbo.DimMes eliminada
  ... (6 dimensiones más) ...
  → Eliminando Tabla de staging (sin FK) (dbo.StgSesiones)
    ✓ dbo.StgSesiones eliminada

--- CREANDO TABLA DE STAGING ---
  → Creando dbo.StgSesiones
    ✓ dbo.StgSesiones creada
  → Ajustando tipo de BounceRates a NVARCHAR(32)
  ...

--- CREANDO DIMENSIONES ---
  → Creando dbo.DimMes
    ✓ dbo.DimMes creada (PK=MesKey, UNIQUE=Mes)
  ... (5 dimensiones más) ...

--- CREANDO TABLA DE HECHOS ---
  → Creando dbo.FactSesion
    ✓ dbo.FactSesion creada (PK=SesionKey, 6 FK, medidas e indicadores)

--- VALIDACIÓN FINAL ---
Tablas creadas (8): DimMes, DimNavegador, DimRegion, DimSistemaOperativo, DimTipoTrafico, DimTipoVisitante, FactSesion, StgSesiones
Foreign Keys en FactSesion (6):
  - FK_FactSesion_DimMes → DimMes
  - FK_FactSesion_DimNavegador → DimNavegador
  ...
PK y UNIQUE constraints:
  - DimMes.PK_DimMes (PK)
  - DimMes.UQ_DimMes_Mes (UNIQUE)
  ...

======================================================================
PROCESO COMPLETADO CON ÉXITO
======================================================================
```

---

## 6. Verificación manual (opcional)

Si quieres comprobar por tu cuenta desde SSMS o `sqlcmd`:

```sql
-- Listar tablas
SELECT name FROM sys.tables WHERE schema_id = SCHEMA_ID('dbo') ORDER BY name;

-- Ver FK de FactSesion
SELECT fk.name, tr.name AS referencia
FROM sys.foreign_keys fk
JOIN sys.tables tr ON fk.referenced_object_id = tr.object_id
WHERE OBJECT_NAME(fk.parent_object_id) = 'FactSesion';

-- Ver PK y UNIQUE
SELECT t.name, i.name, i.is_unique
FROM sys.indexes i
JOIN sys.tables t ON i.object_id = t.object_id
WHERE t.name IN ('DimMes','DimTipoVisitante','DimSistemaOperativo',
                 'DimNavegador','DimRegion','DimTipoTrafico','FactSesion')
  AND (i.is_primary_key = 1 OR i.is_unique = 1);
```

---

## 7. Estructura de las tablas (resumen)

| Tabla | Filas esperadas | PK | FK | UNIQUE | Notas |
|-------|----------------|-----|----|--------|-------|
| `StgSesiones` | 12,205 (carga) | No | No | No | Recibe CSV tal cual; `NVARCHAR` para booleanos y tasas |
| `DimMes` | 10 | `MesKey` | No | `Mes` | Abreviatura 3 letras |
| `DimTipoVisitante` | 3 | `TipoVisitanteKey` | No | `TipoVisitante` | New/Returning/Other |
| `DimSistemaOperativo` | 8 | `SistemaOperativoKey` | No | `CodigoSistemaOperativo` | Códigos 1–8 |
| `DimNavegador` | 13 | `NavegadorKey` | No | `CodigoNavegador` | Códigos 1–13 |
| `DimRegion` | 9 | `RegionKey` | No | `CodigoRegion` | Códigos 1–9 |
| `DimTipoTrafico` | 20 | `TipoTraficoKey` | No | `CodigoTipoTrafico` | Códigos 1–20 |
| `FactSesion` | 12,205 | `SesionKey` (BIGINT) | 6 | No | Medidas + indicadores |

---

## 8. Solución de problemas comunes

| Error | Causa | Solución |
|-------|-------|----------|
| `Unable to complete login process` | LocalDB no está corriendo | `sqllocaldb start MSSQLLocalDB` |
| `Database 'ECBD_Compradores' does not exist` | Base no creada | Ejecutar `sql/ddl/01_crear_base_y_datamart.sql` |
| `ModuleNotFoundError: pyodbc` | Falta driver | `pip install pyodbc` |
| `Foreign key constraint` al borrar | Orden incorrecto | El script ya usa el orden correcto (hecho → dims → staging) |

---

## 9. Referencia al script original

Este notebook es la versión en **Python** del script SQL:
`sql/ddl/00_reiniciar_datamart.sql`

Ambos hacen **exactamente lo mismo**; elige el que prefieras:
- **SQL**: ejecutar desde SSMS (F5), ideal para DBAs.
- **Python**: ejecutar desde terminal/CI/CD/notebooks, ideal para automatización.

---

> **Nota para el profesor:** Este notebook y su script asociado permiten
> reiniciar el Data Mart de forma controlada y repetible durante el
> desarrollo del ETL en SSIS, sin afectar la base de datos completa
> ni los datos originales en `data/raw/`.